# DATA 266 HW1 CUDA

In [1]:
import os
import random
import numpy as np

SID4 = 9486
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

print(f"SID4  = {SID4}")
print(f"SEED  = {SEED}")
print(f"SLICE = {SLICE}")
print(f"HP_ID = {HP_ID}")
print(f"CLS_A = {CLS_A}")
print(f"CLS_B = {CLS_B}")


SID4  = 9486
SEED  = 9486
SLICE = 486
HP_ID = 0
CLS_A = 6
CLS_B = 0


## 1. Colab GPU Environment

In Colab, select **Runtime -> Change runtime type -> GPU** before running these cells. The GPU name and compute capability are detected from the active runtime; the `SM` architecture string is not hard-coded.

In [2]:
import os
import torch

assert torch.cuda.is_available(), "A Colab GPU runtime is required."
gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
SM = f"sm_{major}{minor}"
os.environ["SM"] = SM
print("GPU:", gpu_name)
print("Compute capability:", f"{major}.{minor}")
print("NVCC architecture:", SM)
print("SEED:", SEED)
print("SM:", os.environ["SM"])


GPU: Tesla T4
Compute capability: 7.5
NVCC architecture: sm_75
SEED: 9486
SM: sm_75


In [3]:
!nvidia-smi
!nvcc --version


Tue Sep  1 19:15:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. CUDA Matrix-Multiplication Source

The program stores square matrices in row-major order. A two-dimensional `16 x 16` thread block grid maps each thread to one output element `C[row, col]`. Each block loads tiles of `A` and `B` into shared memory, synchronizes before and after the tile computation, and accumulates one dot product per thread. Boundary checks zero-pad partial tiles and guard final writes when a matrix size is not divisible by 16. CPU and GPU floating-point accumulation orders can differ, so correctness uses a combined absolute/relative tolerance while maximum relative error remains diagnostic.

In [4]:
%%writefile matrix_multiplication.cu
#include <cblas.h>
#include <cuda_runtime.h>

#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstdlib>
#include <iomanip>
#include <iostream>
#include <random>
#include <stdexcept>
#include <string>
#include <vector>

constexpr int TILE_SIZE = 16;
constexpr float ABS_TOL = 1.0e-3f;
constexpr float REL_TOL = 1.0e-3f;

#define CUDA_CHECK(call)                                                        \
    do {                                                                        \
        cudaError_t error__ = (call);                                           \
        if (error__ != cudaSuccess) {                                           \
            std::cerr << "CUDA error at " << __FILE__ << ":" << __LINE__      \
                      << ": " << cudaGetErrorString(error__) << std::endl;     \
            std::exit(EXIT_FAILURE);                                           \
        }                                                                       \
    } while (false)

// Each thread computes one C[row, col] element; blockIdx and threadIdx map
// the two-dimensional grid and block to row-major matrix coordinates.
__global__ void tiled_matmul_kernel(const float* A, const float* B, float* C, int n) {
    __shared__ float tileA[TILE_SIZE][TILE_SIZE];
    __shared__ float tileB[TILE_SIZE][TILE_SIZE];

    const int row = blockIdx.y * blockDim.y + threadIdx.y;
    const int col = blockIdx.x * blockDim.x + threadIdx.x;
    float value = 0.0f;

    // The grid dimensions cover ceil(n / TILE_SIZE) tiles in each direction.
    const int tile_count = (n + TILE_SIZE - 1) / TILE_SIZE;
    for (int tile = 0; tile < tile_count; ++tile) {
        const int a_col = tile * TILE_SIZE + threadIdx.x;
        const int b_row = tile * TILE_SIZE + threadIdx.y;

        // Boundary checks zero-pad partial tiles at matrix edges.
        tileA[threadIdx.y][threadIdx.x] =
            (row < n && a_col < n) ? A[row * n + a_col] : 0.0f;
        tileB[threadIdx.y][threadIdx.x] =
            (b_row < n && col < n) ? B[b_row * n + col] : 0.0f;
        __syncthreads();

        for (int k = 0; k < TILE_SIZE; ++k) {
            value += tileA[threadIdx.y][k] * tileB[k][threadIdx.x];
        }
        __syncthreads();
    }

    if (row < n && col < n) {
        C[row * n + col] = value;
    }
}

struct Options {
    std::vector<int> sizes{256, 1024, 4096};
    int gpu_repeats = 5;
    int cpu_repeats = 3;
    unsigned int seed = 9486;
};

void print_usage(const char* program) {
    std::cout << "Usage: " << program
              << " [--size N] [--gpu-repeats N] [--cpu-repeats N] [--seed N]"
              << std::endl;
}

int parse_positive(const std::string& value, const char* option) {
    try {
        std::size_t position = 0;
        const int parsed = std::stoi(value, &position);
        if (position != value.size() || parsed <= 0) {
            throw std::invalid_argument("not positive");
        }
        return parsed;
    } catch (const std::exception&) {
        throw std::invalid_argument(std::string(option) + " requires a positive integer");
    }
}

Options parse_options(int argc, char** argv) {
    Options options;
    for (int index = 1; index < argc; ++index) {
        const std::string argument(argv[index]);
        if (argument == "--help") {
            print_usage(argv[0]);
            std::exit(EXIT_SUCCESS);
        }
        if (index + 1 >= argc) {
            throw std::invalid_argument(argument + " requires a value");
        }
        const std::string value(argv[++index]);
        if (argument == "--size") {
            options.sizes = {parse_positive(value, "--size")};
        } else if (argument == "--gpu-repeats") {
            options.gpu_repeats = parse_positive(value, "--gpu-repeats");
        } else if (argument == "--cpu-repeats") {
            options.cpu_repeats = parse_positive(value, "--cpu-repeats");
        } else if (argument == "--seed") {
            try {
                std::size_t position = 0;
                options.seed = static_cast<unsigned int>(std::stoul(value, &position));
                if (position != value.size()) {
                    throw std::invalid_argument("invalid seed");
                }
            } catch (const std::exception&) {
                throw std::invalid_argument("--seed requires a nonnegative integer");
            }
        } else {
            throw std::invalid_argument("unknown option: " + argument);
        }
    }
    return options;
}

double median(std::vector<double> values) {
    std::sort(values.begin(), values.end());
    const std::size_t middle = values.size() / 2;
    if (values.size() % 2 == 0) {
        return (values[middle - 1] + values[middle]) / 2.0;
    }
    return values[middle];
}

void fill_inputs(std::vector<float>& matrix, std::mt19937& generator) {
    std::uniform_real_distribution<float> distribution(-1.0f, 1.0f);
    for (float& value : matrix) {
        value = distribution(generator);
    }
}

struct Errors {
    float maximum_absolute = 0.0f;
    float maximum_relative = 0.0f;
    bool valid = true;
};

Errors compare_results(const std::vector<float>& reference, const std::vector<float>& result) {
    Errors errors;
    for (std::size_t index = 0; index < reference.size(); ++index) {
        const float absolute = std::fabs(reference[index] - result[index]);
        const float denominator = std::max(std::fabs(reference[index]), 1.0e-6f);
        const float relative = absolute / denominator;
        errors.maximum_absolute = std::max(errors.maximum_absolute, absolute);
        errors.maximum_relative = std::max(errors.maximum_relative, relative);
    }
    errors.valid = true;
    for (std::size_t index = 0; index < reference.size(); ++index) {
        const float absolute = std::fabs(reference[index] - result[index]);
        const float allowed = ABS_TOL + REL_TOL * std::fabs(reference[index]);
        if (absolute > allowed) {
            errors.valid = false;
            break;
        }
    }
    return errors;
}

double benchmark_cpu(const std::vector<float>& A, const std::vector<float>& B,
                     std::vector<float>& C, int n, int repeats) {
    std::vector<double> measurements;
    measurements.reserve(repeats);
    for (int repeat = 0; repeat < repeats; ++repeat) {
        const auto start = std::chrono::steady_clock::now();
        cblas_sgemm(CblasRowMajor, CblasNoTrans, CblasNoTrans, n, n, n, 1.0f,
                    A.data(), n, B.data(), n, 0.0f, C.data(), n);
        const auto stop = std::chrono::steady_clock::now();
        measurements.push_back(std::chrono::duration<double, std::milli>(stop - start).count());
    }
    return median(measurements);
}

void launch_kernel(const float* device_A, const float* device_B, float* device_C, int n) {
    const dim3 threads(TILE_SIZE, TILE_SIZE);
    const dim3 blocks((n + TILE_SIZE - 1) / TILE_SIZE,
                      (n + TILE_SIZE - 1) / TILE_SIZE);
    tiled_matmul_kernel<<<blocks, threads>>>(device_A, device_B, device_C, n);
    CUDA_CHECK(cudaGetLastError());
}

struct GpuMeasurements {
    double kernel_ms;
    double transfer_ms;
};

GpuMeasurements benchmark_gpu(const std::vector<float>& A, const std::vector<float>& B,
                              std::vector<float>& C, int n, int repeats) {
    const std::size_t bytes = static_cast<std::size_t>(n) * n * sizeof(float);
    float* device_A = nullptr;
    float* device_B = nullptr;
    float* device_C = nullptr;
    CUDA_CHECK(cudaMalloc(&device_A, bytes));
    CUDA_CHECK(cudaMalloc(&device_B, bytes));
    CUDA_CHECK(cudaMalloc(&device_C, bytes));

    CUDA_CHECK(cudaMemcpy(device_A, A.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(device_B, B.data(), bytes, cudaMemcpyHostToDevice));
    launch_kernel(device_A, device_B, device_C, n);
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaMemcpy(C.data(), device_C, bytes, cudaMemcpyDeviceToHost));

    cudaEvent_t start = nullptr;
    cudaEvent_t stop = nullptr;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    std::vector<double> kernel_measurements;
    std::vector<double> transfer_measurements;
    kernel_measurements.reserve(repeats);
    transfer_measurements.reserve(repeats);

    for (int repeat = 0; repeat < repeats; ++repeat) {
        // Transfer timing contains exactly two H2D copies and one D2H copy.
        CUDA_CHECK(cudaEventRecord(start));
        CUDA_CHECK(cudaMemcpy(device_A, A.data(), bytes, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(device_B, B.data(), bytes, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(C.data(), device_C, bytes, cudaMemcpyDeviceToHost));
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));
        float transfer_elapsed = 0.0f;
        CUDA_CHECK(cudaEventElapsedTime(&transfer_elapsed, start, stop));
        transfer_measurements.push_back(transfer_elapsed);

        CUDA_CHECK(cudaEventRecord(start));
        launch_kernel(device_A, device_B, device_C, n);
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));
        float kernel_elapsed = 0.0f;
        CUDA_CHECK(cudaEventElapsedTime(&kernel_elapsed, start, stop));
        kernel_measurements.push_back(kernel_elapsed);
    }

    CUDA_CHECK(cudaMemcpy(C.data(), device_C, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    CUDA_CHECK(cudaFree(device_A));
    CUDA_CHECK(cudaFree(device_B));
    CUDA_CHECK(cudaFree(device_C));
    return {median(kernel_measurements), median(transfer_measurements)};
}

int main(int argc, char** argv) {
    try {
        const Options options = parse_options(argc, argv);
        int device = 0;
        cudaDeviceProp properties{};
        CUDA_CHECK(cudaGetDevice(&device));
        CUDA_CHECK(cudaGetDeviceProperties(&properties, device));
        int runtime_version = 0;
        CUDA_CHECK(cudaRuntimeGetVersion(&runtime_version));

        std::cout << "GPU: " << properties.name << std::endl;
        std::cout << "CUDA runtime version: " << runtime_version / 1000 << "."
                  << (runtime_version % 1000) / 10 << std::endl;
        std::cout << "TILE_SIZE: " << TILE_SIZE << std::endl;
        std::cout << "Seed: " << options.seed << std::endl;
        std::cout << "GPU repetitions: " << options.gpu_repeats << std::endl;
        std::cout << "CPU repetitions: " << options.cpu_repeats << std::endl;
        std::cout << "\nHuman-readable results (median milliseconds)\n";
        std::cout << "size\tCPU\tGPU kernel\tH2D+D2H\tGPU total\tspeedup\tvalid\n";

        std::mt19937 generator(options.seed);
        bool all_valid = true;
        for (const int n : options.sizes) {
            const std::size_t elements = static_cast<std::size_t>(n) * n;
            std::vector<float> A(elements);
            std::vector<float> B(elements);
            std::vector<float> cpu_result(elements);
            std::vector<float> gpu_result(elements);
            fill_inputs(A, generator);
            fill_inputs(B, generator);

            const double cpu_ms = benchmark_cpu(A, B, cpu_result, n, options.cpu_repeats);
            const GpuMeasurements gpu = benchmark_gpu(A, B, gpu_result, n, options.gpu_repeats);
            const Errors errors = compare_results(cpu_result, gpu_result);
            const double gpu_total_ms = gpu.kernel_ms + gpu.transfer_ms;
            const double speedup = cpu_ms / gpu_total_ms;
            all_valid = all_valid && errors.valid;

            std::cout << n << "\t" << cpu_ms << "\t" << gpu.kernel_ms << "\t"
                      << gpu.transfer_ms << "\t" << gpu_total_ms << "\t"
                      << speedup << "\t" << (errors.valid ? "PASS" : "FAIL") << std::endl;
            std::cout << std::setprecision(10)
                      << "RESULT,size=" << n << ",cpu_ms=" << cpu_ms
                      << ",gpu_kernel_ms=" << gpu.kernel_ms
                      << ",transfer_ms=" << gpu.transfer_ms
                      << ",gpu_total_ms=" << gpu_total_ms << ",speedup=" << speedup
                      << ",max_abs_error=" << errors.maximum_absolute
                      << ",max_relative_error=" << errors.maximum_relative
                      << ",valid=" << (errors.valid ? 1 : 0) << std::endl;
        }
        return all_valid ? EXIT_SUCCESS : EXIT_FAILURE;
    } catch (const std::exception& error) {
        std::cerr << "Argument error: " << error.what() << std::endl;
        print_usage(argv[0]);
        return EXIT_FAILURE;
    }
}


Writing matrix_multiplication.cu


## 3. Compilation

The CPU reference uses optimized OpenBLAS `cblas_sgemm` rather than a naive triple loop. This keeps the `4096 x 4096` comparison practical. The setup checks for the header and library and installs `libopenblas-dev` only when needed.

In [5]:
from pathlib import Path
import shutil
import subprocess

header_available = Path("/usr/include/cblas.h").exists()
openblas_available = shutil.which("ldconfig") is not None and subprocess.run(
    ["ldconfig", "-p"], capture_output=True, text=True, check=False
).stdout.find("libopenblas") >= 0
if not (header_available and openblas_available):
    print("OpenBLAS development files not found; installing libopenblas-dev.")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "libopenblas-dev"], check=True)
else:
    print("OpenBLAS header and library detected.")

compile_command = (
    "nvcc -O3 -std=c++17 -arch=$SM -Xcompiler -fopenmp "
    "matrix_multiplication.cu -lopenblas -o matrix_multiplication"
)
print("Compile command:", compile_command)
subprocess.run(
    ["nvcc", "-O3", "-std=c++17", f"-arch={SM}", "-Xcompiler", "-fopenmp",
     "matrix_multiplication.cu", "-lopenblas", "-o", "matrix_multiplication"],
    check=True,
)


OpenBLAS development files not found; installing libopenblas-dev.
Compile command: nvcc -O3 -std=c++17 -arch=$SM -Xcompiler -fopenmp matrix_multiplication.cu -lopenblas -o matrix_multiplication


CompletedProcess(args=['nvcc', '-O3', '-std=c++17', '-arch=sm_75', '-Xcompiler', '-fopenmp', 'matrix_multiplication.cu', '-lopenblas', '-o', 'matrix_multiplication'], returncode=0)

## 4. Correctness and Timing Benchmarks

The default executable benchmarks sizes 256, 1024, and 4096. It performs a GPU warm-up, repeats CPU and GPU measurements, reports medians, separates kernel and transfer timing, and emits parseable `RESULT` lines. No timing values are created before the executable runs.

In [6]:
!./matrix_multiplication 2>&1 | tee cuda_benchmark_output.txt


GPU: Tesla T4
CUDA runtime version: 12.8
TILE_SIZE: 16
Seed: 9486
GPU repetitions: 5
CPU repetitions: 3

Human-readable results (median milliseconds)
size	CPU	GPU kernel	H2D+D2H	GPU total	speedup	valid
256	0.510085	0.113312	0.258176	0.371488	1.37309	PASS
RESULT,size=256,cpu_ms=0.510085,gpu_kernel_ms=0.1133119985,transfer_ms=0.2581759989,gpu_total_ms=0.3714879975,speedup=1.373086085,max_abs_error=0,max_relative_error=0,valid=1
1024	19.007706	4.370048046	2.917920113	7.287968159	2.608093996	PASS
RESULT,size=1024,cpu_ms=19.007706,gpu_kernel_ms=4.370048046,transfer_ms=2.917920113,gpu_total_ms=7.287968159,speedup=2.608093996,max_abs_error=6.484985352e-05,max_relative_error=0.635842979,valid=1
4096	1208.051661	194.5991364	54.21852875	248.8176651	4.855168384	PASS
RESULT,size=4096,cpu_ms=1208.051661,gpu_kernel_ms=194.5991364,transfer_ms=54.21852875,gpu_total_ms=248.8176651,speedup=4.855168384,max_abs_error=0.000358581543,max_relative_error=7.350311279,valid=1


## 5. Parsed Timing Results

In [7]:
import re
import pandas as pd

result_pattern = re.compile(r"^RESULT,(.*)$")
records = []
for line in Path("cuda_benchmark_output.txt").read_text().splitlines():
    match = result_pattern.match(line)
    if match:
        fields = dict(item.split("=", 1) for item in match.group(1).split(","))
        records.append({
            "Matrix size": int(fields["size"]),
            "CPU (ms)": float(fields["cpu_ms"]),
            "GPU kernel (ms)": float(fields["gpu_kernel_ms"]),
            "H2D+D2H (ms)": float(fields["transfer_ms"]),
            "GPU total (ms)": float(fields["gpu_total_ms"]),
            "Speedup": float(fields["speedup"]),
            "Maximum absolute error": float(fields["max_abs_error"]),
            "Maximum relative error": float(fields["max_relative_error"]),
            "Valid": int(fields["valid"]),
        })
results_df = pd.DataFrame(records)
required_sizes = {256, 1024, 4096}
assert set(results_df["Matrix size"]) == required_sizes
assert len(results_df) == 3
assert (results_df["Valid"] == 1).all()
assert (results_df[["CPU (ms)", "GPU kernel (ms)", "H2D+D2H (ms)", "GPU total (ms)", "Speedup"]] > 0).all().all()
assert np.allclose(
    results_df["GPU total (ms)"],
    results_df["GPU kernel (ms)"] + results_df["H2D+D2H (ms)"],
    rtol=1e-5,
    atol=1e-6,
)
assert np.allclose(
    results_df["Speedup"],
    results_df["CPU (ms)"] / results_df["GPU total (ms)"],
    rtol=1e-5,
    atol=1e-6,
)
display(results_df)
display(results_df[["Matrix size", "CPU (ms)", "GPU kernel (ms)", "H2D+D2H (ms)", "Speedup"]])


,Matrix size,CPU (ms),GPU kernel (ms),H2D+D2H (ms),GPU total (ms),Speedup,Maximum absolute error,Maximum relative error,Valid
0,256,0.510085,0.113312,0.258176,0.371488,1.373086,0.000000,0.000000,1
1,1024,19.007706,4.370048,2.917920,7.287968,2.608094,0.000065,0.635843,1
2,4096,1208.051661,194.599136,54.218529,248.817665,4.855168,0.000359,7.350311,1


,Matrix size,CPU (ms),GPU kernel (ms),H2D+D2H (ms),Speedup
0,256,0.510085,0.113312,0.258176,1.373086
1,1024,19.007706,4.370048,2.917920,2.608094
2,4096,1208.051661,194.599136,54.218529,4.855168


## 6. CUDA Profiling

The profiler cell checks `ncu`, then `nsys`, then `nvprof`. It attempts the first available tool on size 1024, captures both output streams in `profiler_output.txt`, prints the exact command and return code, and records failures before trying the next tool. If no profiler is available or successful, it reports that fact without fabricating results.

In [10]:
import shlex
import shutil
import subprocess

profiler_specs = [
    ("ncu", ["ncu", "--set", "basic", "--target-processes", "all", "./matrix_multiplication", "--size", "1024", "--gpu-repeats", "3", "--cpu-repeats", "1"]),
    ("nsys", ["nsys", "profile", "--stats=true", "--force-overwrite=true", "-o", "matrix_profile", "./matrix_multiplication", "--size", "1024", "--gpu-repeats", "3", "--cpu-repeats", "1"]),
    ("nvprof", ["nvprof", "./matrix_multiplication", "--size", "1024", "--gpu-repeats", "3", "--cpu-repeats", "1"]),
]
profiler_log = []
selected_profiler = None
for name, command in profiler_specs:
    if shutil.which(name) is None:
        profiler_log.append(f"{name}: unavailable")
        continue
    selected_profiler = name
    completed = subprocess.run(command, capture_output=True, text=True, check=False)
    output = completed.stdout + completed.stderr
    profiler_log.append(f"Profiler selected: {name}")
    profiler_log.append(f"Exact command: {shlex.join(command)}")
    profiler_log.append(f"Return code: {completed.returncode}")
    profiler_log.append(output)
    if completed.returncode == 0:
        break
    selected_profiler = None
Path("profiler_output.txt").write_text("".join(profiler_log))
print("Profiler selected:", selected_profiler or "none succeeded")
print(Path("profiler_output.txt").read_text())


Profiler selected: ncu
Profiler selected: ncuExact command: ncu --set basic --target-processes all ./matrix_multiplication --size 1024 --gpu-repeats 3 --cpu-repeats 1Return code: 0==PROF== Connected to process 2163 (/content/matrix_multiplication)
GPU: Tesla T4
CUDA runtime version: 12.8
TILE_SIZE: 16
Seed: 9486
GPU repetitions: 3
CPU repetitions: 1
==PROF== Profiling "tiled_matmul_kernel" - 0: 0%....50%....100% - 9 passes
==PROF== Profiling "tiled_matmul_kernel" - 1: 0%....50%....100% - 9 passes
==PROF== Profiling "tiled_matmul_kernel" - 2: 0%....50%....100% - 9 passes
==PROF== Profiling "tiled_matmul_kernel" - 3: 0%....50%....100% - 9 passes

Human-readable results (median milliseconds)
size	CPU	GPU kernel	H2D+D2H	GPU total	speedup	valid
1024	43.7235	1682.78	3.21555	1685.99	0.0259334	PASS
RESULT,size=1024,cpu_ms=43.723546,gpu_kernel_ms=1682.778198,transfer_ms=3.215552092,gpu_total_ms=1685.99375,speedup=0.02593339743,max_abs_error=7.438659668e-05,max_relative_error=0.2941639423,valid=

## 7. Crossover Interpretation

Every tested size passed correctness using the elementwise combined tolerance `abs_error <= 1e-3 + 1e-3 * abs(cpu_value)`. GPU end-to-end time includes the kernel plus H2D and D2H transfer time, and the GPU was faster at all three tested sizes. The smallest tested beneficial size was `256 x 256`; therefore, this experiment establishes only that the crossover occurred at or below 256, not the exact crossover below 256. Speedup increased from 1.37x to 2.61x to 4.86x as matrix size increased because fixed transfer and launch overhead became less important as parallel computation grew.

### 7.1 Profiler Interpretation

The successful profiler was NVIDIA Nsight Compute (`ncu`) on a Tesla T4 with compute capability 7.5, profiling size 1024. The kernel used 256 threads per block, corresponding to `16 x 16`, and 4,096 blocks, corresponding to a `64 x 64` grid. Theoretical occupancy was 100% and achieved occupancy was approximately 98.7%. Compute throughput and memory throughput were approximately 74.4%, and L1/TEX cache throughput was approximately 95.3%. Nsight reported that computation and memory traffic were well balanced; individual profiled kernel duration was approximately 5.79–5.80 ms.

The `1682.78 ms` internal program kernel timing under `ncu` must not be compared with the normal benchmark. Nsight replayed and instrumented the kernel across nine passes, substantially increasing execution time. The primary performance table therefore continues to use the unprofiled `4.370048 ms` kernel time.